# Train a real custom neural ONNX voice

This notebook builds a **fine-tuned pretrained full LLVC streaming student** using real GPU gradient training, not the prior 65–70 KB Pocket adapter. You supply **one voice recording** that you own or have permission to use. It automatically converts public source speech through a reference-conditioned Seed-VC teacher, builds aligned original/converted pairs, fine-tunes the official pretrained KoeAI LLVC HiFi-GAN architecture, exports ONNX with `audio_in → audio_out`, and tests ONNX Runtime inference.

**Requirements:** Google Colab runtime with an NVIDIA GPU (Runtime → Change runtime type → GPU); a clear voice sample, ideally 15–30 seconds, without music. All generated clips and model files are stored temporarily in the Colab runtime; download the final ONNX. This is **not** on-device training on your Chromebook. Training and model downloads need GPU, storage, and network access. Colab resources and runtime duration are not guaranteed.

**How to use:** Run the following cells in order. No manual source/target matching or separate RVC training is required. The teacher itself is a pretrained zero-shot converter that uses your sample; the resulting LLVC model is trained to imitate it.

**Verification distinction:** Successful export means a full neural network trained and executes. Only listening tests and target-speaker similarity evaluation establish voice quality. Do not treat an ONNX checksum or filesize as proof that it sounds right.

In [ ]:
import torch
assert torch.cuda.is_available(), "Choose Runtime → Change runtime type → GPU before proceeding."
print("GPU:", torch.cuda.get_device_name(0))
import os, subprocess, sys
from pathlib import Path
def run(*args, cwd=None):
    print("+", " ".join(str(a) for a in args), flush=True)
    subprocess.run([str(a) for a in args], cwd=cwd, check=True)
os.chdir("/content")
for folder, url in (
    ("irl-voice", "https://github.com/tbenitz/IRL-Voice-Changer.git"),
    ("seed-vc", "https://github.com/Plachtaa/seed-vc.git"),
    ("llvc-upstream", "https://github.com/KoeAI/LLVC.git"),
):
    if not Path(folder).is_dir():
        run("git", "clone", "--depth", "1", url, folder)
run("apt-get", "update", "-qq")
run("apt-get", "install", "-y", "-qq", "ffmpeg")
# Deliberately avoid the upstream legacy fairseq and Gradio training packages:
# the full LLVC generator/discriminator still trains with auxiliary mel loss.
run(sys.executable, "-m", "pip", "install", "-q",
    "numpy<2", "librosa==0.10.2", "soundfile==0.12.1",
    "transformers==4.46.3", "huggingface-hub>=0.28.1,<1",
    "munch==4.0.0", "einops==0.8.0", "accelerate",
    "descript-audio-codec==1.0.0", "funasr==1.1.5", "modelscope==1.18.1",
    "hydra-core", "resemblyzer", "jiwer", "pydub",
    "speechbrain", "auraloss", "torchmetrics", "tensorboard",
    "onnx", "onnxruntime", "safetensors", "sentencepiece")
print("Setup complete; Colab may request a runtime restart after package installation.")

## 1. Upload ONE reference voice

Upload a clean WAV, MP3, M4A, or FLAC of the voice to be modeled. The file is sent to your Colab runtime, not to GitHub. Use a recording you have permission to process.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one speaker reference file"
name = next(iter(uploaded))
reference = "/content/target_reference.wav"
run("ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-i", "/content/" + name, "-ac", "1", "-ar", "22050",
    "-t", "30", "-acodec", "pcm_s16le", reference)
import soundfile as sf
y, sr = sf.read(reference)
assert sr == 22050 and len(y) >= sr * 3, "Need at least 3 seconds of usable target voice"
print("Target reference:", round(len(y)/sr, 1), "seconds; no other user audio is needed")

## 2. Fetch neutral speech and automatically generate real teacher-converted pairs

This uses the LibriSpeech dev-clean corpus (a separate public speech dataset, not your recording) and Seed-VC as an actual reference-conditioned voice-conversion teacher. Unlike two Pocket TTS sentences, each pair is an original recording and a teacher conversion **of that same recording**.

In [ ]:
archive = Path("/content/dev-clean.tar.gz")
if not Path("/content/LibriSpeech/dev-clean").is_dir():
    if not archive.exists():
        run("wget", "-q", "--show-progress", "-O", archive,
            "https://www.openslr.org/resources/12/dev-clean.tar.gz")
    run("tar", "-xzf", archive, "-C", "/content")
# Increase PAIRS to 500 for the larger dataset used in published LLVC workflows.
PAIRS = 300
run(sys.executable,
    "/content/irl-voice/voice-training/generate_teacher_pairs.py",
    "--reference", reference,
    "--source-dir", "/content/LibriSpeech/dev-clean",
    "--output", "/content/custom-voice/dataset",
    "--seed-vc", "/content/seed-vc",
    "--pairs", str(PAIRS),
    "--teacher-steps", "10")
print(Path("/content/custom-voice/dataset/manifest.json").read_text()[:1200])

## 3. Train the full LLVC model (not the small Pocket student)

This **fine-tunes the official KoeAI pretrained LLVC HiFi-GAN generator** (not random weights or a 70 KB adapter) using real gradient updates on teacher-converted speech. It saves checkpoints every 500 updates. Higher-quality conversions generally require a substantial paired dataset and extensive GPU training; default values are a starting point, not a guaranteed quality level.

In [ ]:
STEPS = 5000   # Change to 10000+ for a longer training run if GPU session permits.
BATCH = 2
run(sys.executable,
    "/content/irl-voice/voice-training/train_full_llvc.py",
    "--llvc", "/content/llvc-upstream",
    "--dataset", "/content/custom-voice/dataset",
    "--output", "/content/custom-voice/llvc-trained",
    "--steps", str(STEPS),
    "--batch", str(BATCH),
    "--name", "my_custom_neural_voice")

## 4. Download and test the actual ONNX

The exporter checks that the ONNX file contains a full trained generator, uses the correct 16 kHz `audio_in` / `audio_out` interface, executes in ONNX Runtime, and produces finite audio at several chunk sizes. Import the ONNX file into **Fast Neural → Import ONNX** on the original IRL Voice Changer site. Listen and benchmark it on your Chromebook. If training or export fails, this cell will not invent or substitute a fake model.

In [ ]:
from google.colab import files
model = Path("/content/custom-voice/llvc-trained/my_custom_neural_voice.onnx")
report = Path("/content/custom-voice/llvc-trained/model_report.json")
assert model.exists() and report.exists(), "No verified trained model. Check the failed cell above."
print(report.read_text())
files.download(str(model))
files.download(str(report))
for ab_name in ("AB_original.wav", "AB_teacher.wav", "AB_trained_ONNX.wav"):
    ab_path = report.parent / ab_name
    assert ab_path.is_file(), "Missing trained ONNX A/B audition clip: " + ab_name
    files.download(str(ab_path))